# Foundations A3: Matrix Multiplication and Reshaping

Companion notebook for [Foundations A3 of the interactive course](https://bc-sparknet.github.io/make-an-llm-2/#/chapter/matmul). The code cells follow the chapter's examples in order; the explanations are on the site.

*This notebook is generated from the chapter source. To change it, edit the chapter and run `npm run notebooks`.*

In [ ]:
%pip install -q torch tiktoken

## The dot product: two vectors in, one number out

In [ ]:
import torch
import torch.nn as nn

a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])

print(a * b)          # tensor([ 4., 10., 18.])  entry by entry
print((a * b).sum())  # tensor(32.)              then add up
print(a @ b)          # tensor(32.)              @ does both at once

In [ ]:
cat    = torch.tensor([0.9, 0.3, 0.2])
kitten = torch.tensor([0.8, 0.4, 0.1])
car    = torch.tensor([0.1, 0.2, 0.9])

print(cat @ kitten)   # tensor(0.8600)  similar direction: big
print(cat @ car)      # tensor(0.3300)  different direction: small

## Matrix multiplication is a grid of dot products

In [ ]:
A = torch.tensor([[1, 2],
                  [3, 0],
                  [2, 1]])        # 3 × 2
B = torch.tensor([[1, 0, 2],
                  [3, 1, 1]])     # 2 × 3

C = A @ B
print(C.shape)                    # torch.Size([3, 3])
print(C)
# tensor([[7, 2, 4],
#         [3, 0, 6],
#         [5, 1, 5]])

print(A[1] @ B[:, 2])             # tensor(6)  row 1 of A · column 2 of B = C[1, 2]

C_loop = torch.zeros(3, 3, dtype=torch.long)
for i in range(3):
    for j in range(3):
        C_loop[i, j] = A[i] @ B[:, j]
print(torch.equal(C, C_loop))     # True

### The shape-checking habit

In [ ]:
try:
    A @ A                          # (3 × 2) @ (3 × 2): inner dims are 2 and 3
except RuntimeError as e:
    print("Error:", e)
# Error: mat1 and mat2 shapes cannot be multiplied (3x2 and 3x2)

## Transpose: turning rows into columns

In [ ]:
print(A.shape, A.T.shape)                  # torch.Size([3, 2]) torch.Size([2, 3])
print(A.T)
# tensor([[1, 3, 2],
#         [2, 0, 1]])
print(torch.equal(A.T, A.transpose(0, 1)))  # True

### Why attention needs `Q @ K.T`

In [ ]:
torch.manual_seed(0)
Q = torch.randn(5, 8)
K = torch.randn(5, 8)

print((Q @ K.T).shape)    # torch.Size([5, 5])  tokens × tokens

try:
    Q @ K                  # (5 × 8) @ (5 × 8): inner 8 vs 5
except RuntimeError as e:
    print("Error:", e)
# Error: mat1 and mat2 shapes cannot be multiplied (5x8 and 5x8)

## nn.Linear is a matrix multiply

In [ ]:
torch.manual_seed(0)
layer = nn.Linear(4, 3)
print(layer.weight.shape)   # torch.Size([3, 4])
print(layer.bias.shape)     # torch.Size([3])

x = torch.randn(2, 5, 4)    # batch 2, 5 tokens, 4 features each
out = layer(x)
print(out.shape)            # torch.Size([2, 5, 3])

manual = x @ layer.weight.T + layer.bias
print(torch.allclose(out, manual))   # True

In [ ]:
# Token 3 of sequence 1, pushed through the layer on its own, gives the same answer:
print(torch.allclose(out[1, 3], layer(x[1, 3])))   # True

## Batched matmul: @ in more than two dimensions

In [ ]:
X = torch.randn(2, 3, 4)
Y = torch.randn(2, 4, 5)

Z = X @ Y
print(Z.shape)                                   # torch.Size([2, 3, 5])

Z_loop = torch.stack([X[i] @ Y[i] for i in range(2)])
print(torch.allclose(Z, Z_loop))                 # True

In [ ]:
W = torch.randn(4, 5)
print((X @ W).shape)    # torch.Size([2, 3, 5])  the same W for both batch entries

### The attention case: four dimensions

In [ ]:
q = torch.randn(2, 12, 6, 64)       # batch 2, 12 heads, 6 tokens, head_dim 64
k = torch.randn(2, 12, 6, 64)

print(k.transpose(2, 3).shape)      # torch.Size([2, 12, 64, 6])
scores = q @ k.transpose(2, 3)
print(scores.shape)                 # torch.Size([2, 12, 6, 6])

# Batch 1, head 7 is just a plain 2-D Q @ K.T:
print(torch.allclose(scores[1, 7], q[1, 7] @ k[1, 7].T))   # True

In [ ]:
try:
    torch.randn(2, 3, 4) @ torch.randn(3, 4, 5)   # batch dims 2 vs 3
except RuntimeError as e:
    print("Error:", e)
# Error: The size of tensor a (2) must match the size of tensor b (3) at non-singleton dimension 0

## Reshaping: same numbers, new shape

### view and reshape: cutting the same strip differently

In [ ]:
t = torch.arange(12)
print(t)            # tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])

print(t.view(3, 4))
# tensor([[ 0,  1,  2,  3],
#         [ 4,  5,  6,  7],
#         [ 8,  9, 10, 11]])

print(t.view(4, 3))
# tensor([[ 0,  1,  2],
#         [ 3,  4,  5],
#         [ 6,  7,  8],
#         [ 9, 10, 11]])

print(t.view(2, 3, 2).shape)   # torch.Size([2, 3, 2])

In [ ]:
print(t.view(-1, 6).shape)      # torch.Size([2, 6])     12 / 6 = 2
print(t.view(2, -1, 3).shape)   # torch.Size([2, 2, 3])  12 / (2 · 3) = 2

try:
    t.view(5, -1)               # 12 isn't divisible by 5
except RuntimeError as e:
    print("Error:", e)
# Error: shape '[5, -1]' is invalid for input of size 12

In [ ]:
x3 = torch.arange(24).view(2, 3, 4)
print(x3.view(2, 3, 2, 2)[1, 2])    # row x3[1, 2] = [20, 21, 22, 23], folded
# tensor([[20, 21],
#         [22, 23]])

### flatten: merging dims

In [ ]:
logits = torch.randn(2, 3, 50257)            # batch 2, 3 tokens, vocab 50257
targets = torch.randint(0, 50257, (2, 3))

print(logits.flatten(0, 1).shape)            # torch.Size([6, 50257])
print(targets.flatten().shape)               # torch.Size([6])

### transpose and permute: changing the reading order

In [ ]:
m = t.view(3, 4)
print(m.T)
# tensor([[ 0,  4,  8],
#         [ 1,  5,  9],
#         [ 2,  6, 10],
#         [ 3,  7, 11]])

In [ ]:
x = torch.randn(2, 3, 4)
print(x.permute(2, 0, 1).shape)   # torch.Size([4, 2, 3])  old dim 2 first, then 0, then 1
print(x.transpose(0, 2).shape)    # torch.Size([4, 3, 2])  just swap dims 0 and 2

### contiguous(): why view after transpose needs it

In [ ]:
mt = m.T
print(m.is_contiguous(), mt.is_contiguous())   # True False

try:
    mt.view(12)
except RuntimeError as e:
    print("Error:", e)
# Error: view size is not compatible with input tensor's size and stride (...). Use .reshape(...) instead.

print(mt.contiguous().view(12))
# tensor([ 0,  4,  8,  1,  5,  9,  2,  6, 10,  3,  7, 11])
print(mt.reshape(12))   # same result: reshape copies when it has to

## Putting it together: splitting attention heads

In [ ]:
b, n, d_out, num_heads = 1, 4, 6, 2
head_dim = d_out // num_heads            # 3

q = torch.arange(24).view(b, n, d_out)   # stand-in for self.to_q(x)
print(q)
# tensor([[[ 0,  1,  2,  3,  4,  5],
#          [ 6,  7,  8,  9, 10, 11],
#          [12, 13, 14, 15, 16, 17],
#          [18, 19, 20, 21, 22, 23]]])

In [ ]:
q = q.view(b, n, num_heads, head_dim)
print(q.shape)        # torch.Size([1, 4, 2, 3])  [batch, tokens, heads, head_dim]

In [ ]:
q = q.transpose(1, 2)
print(q.shape)        # torch.Size([1, 2, 4, 3])  [batch, heads, tokens, head_dim]

print(q[0, 0])        # head 0: columns 0-2 of every token
# tensor([[ 0,  1,  2],
#         [ 6,  7,  8],
#         [12, 13, 14],
#         [18, 19, 20]])
print(q[0, 1])        # head 1: columns 3-5 of every token
# tensor([[ 3,  4,  5],
#         [ 9, 10, 11],
#         [15, 16, 17],
#         [21, 22, 23]])

In [ ]:
k = q
scores = q @ k.transpose(2, 3)
print(scores.shape)   # torch.Size([1, 2, 4, 4])  a 4 × 4 score grid per head

In [ ]:
heads_out = q.contiguous()           # pretend: weights @ v, a fresh [1, 2, 4, 3] tensor
out = heads_out.transpose(1, 2)
print(out.shape)                     # torch.Size([1, 4, 2, 3])
print(out.is_contiguous())           # False

out = out.contiguous().view(b, n, d_out)
print(out.shape)                     # torch.Size([1, 4, 6])
print(torch.equal(out, torch.arange(24).view(b, n, d_out)))   # True: back where we started

In [ ]:
wrong = heads_out.view(b, n, d_out)  # forgot the transpose: no error!
print(wrong[0, 0])                   # tensor([0, 1, 2, 6, 7, 8])  token 0 + token 1: scrambled

## Summary